# Meet temperatures

Adds a `temp` column to `regular_season.csv` and `bigwest_2021_2026_results.csv`: the temperature where and when each meet was held.

**How:**
1. **Where each meet was held.** Regular-season meets: each athlete's TFRRS page links to every meet they ran, and each meet's TFRRS page lists its venue and city (e.g. "UC Irvine - Irvine, CA"). Championships: the same location line, read from the saved pages in `data/raw/championships`.
2. **City → coordinates** with the [Open-Meteo geocoding API](https://open-meteo.com/en/docs/geocoding-api).
3. **Temperature** from the [Open-Meteo historical weather API](https://open-meteo.com/en/docs/historical-weather-api), a free weather-history service built from weather-model data, not a station at the track.

**What `temp` means:** the day's high temperature in °F, averaged over the meet's days (a 2-day meet gets the mean of its two highs), since we don't know which day each athlete competed.

**Championship dates:** TFRRS lists 2026 as May 8–16 because it folds the multi-events weekend (decathlon/heptathlon, May 8–9 in Irvine) into the main meet (May 15–16 in Long Beach). `CHAMP_DATE_OVERRIDES` uses the main-meet days, so `temp` describes the meet almost every athlete competed in.

**Indoor meets get no temperature** (left blank): the outdoor weather doesn't describe conditions inside. Indoor meets are flagged with the same rule as the predictor notebook.

`meet_weather.csv` keeps one row per meet with its TFRRS link, location, coordinates and `temp`, so every value can be checked.

In [1]:
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

DATA_DIR = Path('../data/processed')
RAW_CHAMPS_DIR = Path('../data/raw/championships')
REGULAR_SEASON_PATH = DATA_DIR / 'regular_season.csv'
RESULTS_PATH = DATA_DIR / 'bigwest_2021_2026_results.csv'
MEET_WEATHER_PATH = DATA_DIR / 'meet_weather.csv'  # one row per meet: location + temp

REFRESH_LOCATIONS = False  # True = look up every meet on TFRRS again, even if meet_weather.csv exists
CACHE_DIR = None           # optional folder to keep downloaded TFRRS pages between runs
PAUSE_SECONDS = 1.0        # wait between TFRRS requests, to go easy on the site

# Main-meet days where the TFRRS header also covers a separate multi-events weekend
# (2026: multis May 8-9 in Irvine, main meet May 15-16 in Long Beach; bigwest.org)
CHAMP_DATE_OVERRIDES = {2026: 'May 15-16, 2026'}

# Read every column as text so the CSVs are written back exactly as they were, plus `temp`
regular_season = pd.read_csv(REGULAR_SEASON_PATH, dtype=str, keep_default_na=False)
results = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
regular_season_columns, results_columns = list(regular_season.columns), list(results.columns)
regular_season = regular_season.drop(columns='temp', errors='ignore')
results = results.drop(columns='temp', errors='ignore')
print(f'{len(regular_season)} regular-season rows, {len(results)} championship rows')

41493 regular-season rows, 4579 championship rows


## Helpers

In [2]:
SESSION = requests.Session()
SESSION.headers['User-Agent'] = 'Mozilla/5.0'


def norm(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def fetch(url, cache_name):
    """GET a TFRRS page, politely, with an optional on-disk cache."""
    cache_path = Path(CACHE_DIR) / cache_name if CACHE_DIR else None
    if cache_path and cache_path.exists():
        return cache_path.read_text(encoding='utf-8')
    for attempt in range(3):
        resp = SESSION.get(url, timeout=30)
        if resp.status_code == 200:
            if cache_path:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_path.write_text(resp.text, encoding='utf-8')
            time.sleep(PAUSE_SECONDS)
            return resp.text
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f'HTTP {resp.status_code} for {url}')


MEET_LINK = re.compile(r'/results/(?:xc/)?(\d+)/[^/#?]+/?$')


def athlete_meets(html):
    """(meet_name, meet_date, tfrrs_meet_id, url) for each meet listed on a TFRRS athlete page."""
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for a in soup.select('a[href*="/results/"]'):
        m = MEET_LINK.search(a['href'])
        if not m:
            continue  # links to a single event, not the meet
        name = norm(a.get_text())
        container = a.find_parent(['th', 'tr', 'div'])
        date = norm(norm(container.get_text(' ')).replace(name, '', 1)) if container else ''
        rows.append((name, date, int(m.group(1)), a['href']))
    return rows


def meet_header(html):
    """Date line and 'Venue - City, ST' line from the header of a TFRRS meet page."""
    soup = BeautifulSoup(html, 'html.parser')
    parts = [norm(el.get_text(' ')) for el in soup.select('.panel-heading-normal-text')]
    parts = [p for p in parts if p and p != '|']
    return {'tfrrs_date': parts[0] if parts else None,
            'location': parts[1] if len(parts) > 1 else None}


def split_location(text):
    """'UC Irvine - Irvine, CA' -> ('UC Irvine', 'Irvine', 'CA')."""
    if not isinstance(text, str):
        return None, None, None
    venue, _, place = text.rpartition(' - ')
    m = re.match(r'^(.+?),\s*([A-Z]{2})$', place.strip())
    if not m:
        return venue or None, None, None
    return venue or None, m.group(1).strip(), m.group(2)


DATE = re.compile(r'^([A-Z][a-z]{2})\s+(\d{1,2})(?:\s*-\s*(?:([A-Z][a-z]{2})\s+)?(\d{1,2}))?,\s*(\d{4})$')


def date_range(text):
    """'Mar 31-Apr 1, 2023' -> (2023-03-31, 2023-04-01)."""
    m = DATE.match(text.strip())
    if not m:
        raise ValueError(f'Unrecognized meet date: {text!r}')
    mon1, day1, mon2, day2, year = m.groups()
    return pd.Timestamp(f'{mon1} {day1} {year}'), pd.Timestamp(f'{mon2 or mon1} {day2 or day1} {year}')


STATES = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'DC': 'District of Columbia',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois',
    'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana',
    'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon',
    'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota',
    'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia',
    'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
}


def geocode(city, state):
    """(latitude, longitude) of a US city, via Open-Meteo's geocoding API."""
    resp = requests.get('https://geocoding-api.open-meteo.com/v1/search',
                        params={'name': city, 'count': 20, 'country_code': 'US'}, timeout=30)
    resp.raise_for_status()
    for r in resp.json().get('results', []):
        if r.get('admin1') == STATES.get(state):
            return r['latitude'], r['longitude']
    return np.nan, np.nan


def daily_highs(lat, lon, start, end):
    """Daily high temperatures (°F) from Open-Meteo's historical weather API."""
    resp = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
        'latitude': lat, 'longitude': lon,
        'start_date': start.strftime('%Y-%m-%d'), 'end_date': end.strftime('%Y-%m-%d'),
        'daily': 'temperature_2m_max', 'temperature_unit': 'fahrenheit', 'timezone': 'auto',
    }, timeout=30)
    resp.raise_for_status()
    return pd.Series(resp.json()['daily']['temperature_2m_max'], dtype=float)

## 1. Where each meet was held

**Regular season:** pick the fewest athletes whose TFRRS pages together list every meet, read each meet's TFRRS link from those pages, then read the location from each meet's page. Skipped if `meet_weather.csv` already exists (set `REFRESH_LOCATIONS = True` to redo it).

In [3]:
meets = regular_season[['meet_name', 'meet_date']].drop_duplicates().reset_index(drop=True)

if MEET_WEATHER_PATH.exists() and not REFRESH_LOCATIONS:
    saved = pd.read_csv(MEET_WEATHER_PATH)
    saved = saved[saved['source'] == 'regular_season']
    season_meets = meets.merge(saved[['meet_name', 'meet_date', 'tfrrs_meet_id', 'tfrrs_url', 'location']],
                               on=['meet_name', 'meet_date'], how='left')
    print(f'Reused saved locations for {season_meets["location"].notna().sum()} of {len(meets)} meets')
else:
    # Fewest athletes whose pages cover every meet (greedy set cover)
    meet_keys = pd.Series(list(zip(regular_season['meet_name'], regular_season['meet_date'])))
    meets_by_athlete = meet_keys.groupby(regular_season['athlete_id'].astype(int).values).apply(set).to_dict()
    uncovered = set(zip(meets['meet_name'], meets['meet_date']))
    athletes = []
    while uncovered:
        best = max(meets_by_athlete, key=lambda a: len(meets_by_athlete[a] & uncovered))
        athletes.append(best)
        uncovered -= meets_by_athlete[best]
    print(f'Reading {len(athletes)} athlete pages to find all {len(meets)} meets')

    links = []
    for athlete_id in tqdm(athletes, desc='Athlete pages'):
        links += athlete_meets(fetch(f'https://www.tfrrs.org/athletes/{athlete_id}.html',
                                     f'athletes/{athlete_id}.html'))
    links = (pd.DataFrame(links, columns=['meet_name', 'date_norm', 'tfrrs_meet_id', 'tfrrs_url'])
             .drop_duplicates(['meet_name', 'date_norm', 'tfrrs_meet_id']))
    season_meets = (meets.assign(date_norm=meets['meet_date'].map(norm))
                    .merge(links, on=['meet_name', 'date_norm'], how='left')
                    .drop(columns='date_norm'))
    assert not season_meets.duplicated(['meet_name', 'meet_date']).any(), 'A meet matched two TFRRS pages'

    found = season_meets.dropna(subset=['tfrrs_meet_id']).drop_duplicates('tfrrs_meet_id')
    print(f'Reading {len(found)} meet pages')
    headers = {row.tfrrs_meet_id: meet_header(fetch(row.tfrrs_url, f'meets/{int(row.tfrrs_meet_id)}.html'))
               for row in tqdm(found.itertuples(), total=len(found), desc='Meet pages')}
    season_meets['location'] = season_meets['tfrrs_meet_id'].map(lambda m: headers.get(m, {}).get('location'))

missing = season_meets[season_meets['location'].isna()]
print(f'{len(season_meets) - len(missing)} of {len(season_meets)} meets have a location')
if len(missing):
    display(missing)

Reused saved locations for 372 of 372 meets
372 of 372 meets have a location


**Championships:** the location line from each year's saved TFRRS page (the men's and women's pages list the same meet).

In [4]:
champ_rows = []
for year in sorted(results['meet_year'].unique()):
    header = meet_header((RAW_CHAMPS_DIR / f'bigwest_{year}_men.html').read_text(encoding='utf-8'))
    women = meet_header((RAW_CHAMPS_DIR / f'bigwest_{year}_women.html').read_text(encoding='utf-8'))
    assert header == women, f'{year}: men and women pages list different dates/locations'
    meet_date = CHAMP_DATE_OVERRIDES.get(int(year), header['tfrrs_date'])
    if meet_date != header['tfrrs_date']:
        print(f"{year}: TFRRS lists {header['tfrrs_date']}; using the main meet, {meet_date}")
    champ_rows.append({'meet_year': year, 'meet_date': meet_date, 'location': header['location']})
champ_meets = pd.DataFrame(champ_rows)
champ_meets

2026: TFRRS lists May 8-16, 2026; using the main meet, May 15-16, 2026


,meet_year,meet_date,location
0,2021,"May 12-15, 2021","UC Irvine - Irvine, CA"
1,2022,"May 13-14, 2022","UC Davis - Davis, CA"
2,2023,"May 12-13, 2023","Cal St. Fullerton - Fullerton, CA"
3,2024,"May 10-11, 2024","Cal Poly-SLO - San Luis Obispo, CA"
4,2025,"May 16-17, 2025","Jack Rose Track - Long Beach, CA"
5,2026,"May 15-16, 2026","Jack Rose Track - Long Beach, CA"


## 2. Indoor meets

Same rule as the predictor notebook: "Indoor" in the name, or no outdoor-only events and either indoor-only events or a January/February date.

In [5]:
INDOOR_ONLY_EVENTS = {'60', '60H', 'WT', 'Pent'}
OUTDOOR_ONLY_EVENTS = {'100', '100H', '110H', '400H', '3000S', '10,000', 'DT', 'HT', 'JT', '4x100', 'Dec'}

events = regular_season.groupby(['meet_name', 'meet_date'])['event'].agg(set)
start_month = regular_season.groupby(['meet_name', 'meet_date'])['meet_start_date'].first().str[5:7].astype(int)
named_indoor = pd.Series(events.index.get_level_values('meet_name').str.contains('indoor', case=False),
                         index=events.index)
has_outdoor = events.map(lambda e: bool(e & OUTDOOR_ONLY_EVENTS))
has_indoor = events.map(lambda e: bool(e & INDOOR_ONLY_EVENTS))
is_indoor = (named_indoor | (~has_outdoor & (has_indoor | (start_month <= 2)))).rename('is_indoor')

season_meets = season_meets.drop(columns='is_indoor', errors='ignore').merge(
    is_indoor.reset_index(), on=['meet_name', 'meet_date'], how='left')
print(season_meets['is_indoor'].value_counts().rename({True: 'indoor', False: 'outdoor'}).to_string())

is_indoor
outdoor    269
indoor     103


## 3. Coordinates and temperature

In [6]:
season_meets['source'] = 'regular_season'
champ_meets['source'] = 'championship'
champ_meets['meet_name'] = 'Big West Championships ' + champ_meets['meet_year']
champ_meets['is_indoor'] = False
meet_weather = pd.concat([season_meets, champ_meets], ignore_index=True)

meet_weather[['venue', 'city', 'state']] = meet_weather['location'].apply(lambda t: pd.Series(split_location(t)))
dates = meet_weather['meet_date'].map(date_range)
meet_weather['start_date'] = [start for start, _ in dates]
meet_weather['end_date'] = [end for _, end in dates]

places = meet_weather[['city', 'state']].dropna().drop_duplicates()
coords = {(c, s): geocode(c, s) for c, s in tqdm(places.itertuples(index=False), total=len(places), desc='Geocoding')}
meet_weather['latitude'] = [coords.get((c, s), (np.nan, np.nan))[0] for c, s in zip(meet_weather['city'], meet_weather['state'])]
meet_weather['longitude'] = [coords.get((c, s), (np.nan, np.nan))[1] for c, s in zip(meet_weather['city'], meet_weather['state'])]
print(f'Geocoded {sum(pd.notna(v[0]) for v in coords.values())} of {len(coords)} cities')

def meet_temp(row):
    if row.is_indoor or pd.isna(row.latitude):
        return np.nan
    return round(daily_highs(row.latitude, row.longitude, row.start_date, row.end_date).mean(), 1)

meet_weather['temp'] = [meet_temp(row) for row in tqdm(meet_weather.itertuples(), total=len(meet_weather), desc='Weather')]

cols = ['source', 'meet_name', 'meet_year', 'meet_date', 'start_date', 'end_date', 'tfrrs_meet_id',
        'tfrrs_url', 'location', 'venue', 'city', 'state', 'latitude', 'longitude', 'is_indoor', 'temp']
meet_weather = meet_weather[cols]
meet_weather.to_csv(MEET_WEATHER_PATH, index=False)
print(meet_weather.groupby('source')['temp'].describe().round(1))

Geocoding:   0%|          | 0/54 [00:00<?, ?it/s]

Geocoded 54 of 54 cities


Weather:   0%|          | 0/378 [00:00<?, ?it/s]

                count  mean   std   min   25%   50%   75%   max
source                                                         
championship      6.0  74.0   8.7  65.2  68.6  72.6  75.5  89.8
regular_season  269.0  71.0  10.1  50.2  63.3  69.9  78.4  98.5


## 4. Add `temp` to the CSVs

In [7]:
def fmt(t):
    return '' if pd.isna(t) else f'{t:.1f}'

season_temp = meet_weather.loc[meet_weather['source'] == 'regular_season', ['meet_name', 'meet_date', 'temp']]
regular_season_out = regular_season.merge(season_temp.assign(temp=season_temp['temp'].map(fmt)),
                                          on=['meet_name', 'meet_date'], how='left', validate='many_to_one')
champ_temp = meet_weather.loc[meet_weather['source'] == 'championship', ['meet_year', 'temp']]
results_out = results.merge(champ_temp.assign(temp=champ_temp['temp'].map(fmt)),
                            on='meet_year', how='left', validate='many_to_one')

# Put `temp` back in its original column position, so a rerun doesn't reorder the CSV
if 'temp' in regular_season_columns:
    regular_season_out = regular_season_out[regular_season_columns]
if 'temp' in results_columns:
    results_out = results_out[results_columns]

assert len(regular_season_out) == len(regular_season) and len(results_out) == len(results)
regular_season_out.to_csv(REGULAR_SEASON_PATH, index=False)
results_out.to_csv(RESULTS_PATH, index=False)

for name, df in [('regular_season.csv', regular_season_out), ('bigwest_2021_2026_results.csv', results_out)]:
    filled = df['temp'].ne('').sum()
    print(f'{name}: temp filled for {filled} of {len(df)} rows ({len(df) - filled} blank)')

regular_season.csv: temp filled for 32771 of 41493 rows (8722 blank)
bigwest_2021_2026_results.csv: temp filled for 4579 of 4579 rows (0 blank)


## Check

In [8]:
display(meet_weather[meet_weather['source'] == 'championship'][['meet_name', 'meet_date', 'location', 'temp']])
outdoor = meet_weather[(meet_weather['source'] == 'regular_season') & ~meet_weather['is_indoor'].astype(bool)]
print('Outdoor regular-season meets with no temp:', outdoor['temp'].isna().sum())
print('Coldest and hottest outdoor meets:')
display(outdoor.dropna(subset=['temp']).sort_values('temp')[['meet_name', 'meet_date', 'location', 'temp']].iloc[[0, 1, 2, -3, -2, -1]])

,meet_name,meet_date,location,temp
372,Big West Championships 2021,"May 12-15, 2021","UC Irvine - Irvine, CA",70.1
373,Big West Championships 2022,"May 13-14, 2022","UC Davis - Davis, CA",89.8
374,Big West Championships 2023,"May 12-13, 2023","Cal St. Fullerton - Fullerton, CA",75.6
375,Big West Championships 2024,"May 10-11, 2024","Cal Poly-SLO - San Luis Obispo, CA",75.2
376,Big West Championships 2025,"May 16-17, 2025","Jack Rose Track - Long Beach, CA",65.2
377,Big West Championships 2026,"May 15-16, 2026","Jack Rose Track - Long Beach, CA",68.1


Outdoor regular-season meets with no temp: 0
Coldest and hottest outdoor meets:


,meet_name,meet_date,location,temp
349,2025 Oregon Preview,"Mar 21-22, 2025","Hayward Field - Eugene, OR",50.2
371,Brutus Hamilton Multi 2024,"Apr 4- 5, 2024","California - Berkeley, CA",52.2
210,Challenge Cup,"Mar 14, 2025","UC Irvine - Irvine, CA",52.3
207,Titan Tune Up,"May 9, 2025","Cal St. Fullerton - Fullerton, CA",96.4
44,UC-Riverside Distance Invite,"Apr 29, 2021","UC Riverside - Riverside, CA",97.6
34,UC-Riverisde Sprint Invitational,"Apr 30, 2021","UC Riverside - Riverside, CA",98.5
